# CS3081 – Assignment: Uncertainty and Bayesian Networks
## Exercise 5(c) – Approximate Inference with pgmpy

**Name:** Noor Abuzinadah  
**Student ID:** s24109202  
**Instructor:** Dr. Naila Marir

### 1. Build the Bayesian network

We recreate the network from Exercise 3:
- Nodes: `heatwave`, `overload`, `powercut`, `cancel`
- Edges: heatwave → overload, heatwave → powercut, overload → powercut, powercut → cancel
- CPTs taken from the assignment tables (only “yes” probabilities shown there; “no” = 1 − yes).

In [1]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling

# Structure
model = DiscreteBayesianNetwork([
    ("heatwave", "overload"),
    ("heatwave", "powercut"),
    ("overload", "powercut"),
    ("powercut", "cancel"),
])

# CPDs  (state order: no=0, yes=1)
cpd_heatwave = TabularCPD(
    variable="heatwave",
    variable_card=2,
    values=[[0.7], [0.3]],          # P(no), P(yes)
    state_names={"heatwave": ["no", "yes"]},
)

cpd_overload = TabularCPD(
    variable="overload",
    variable_card=2,
    values=[
        [0.9, 0.4],                 # P(overload=no | heatwave=no/yes)
        [0.1, 0.6],                 # P(overload=yes | heatwave=no/yes)
    ],
    evidence=["heatwave"],
    evidence_card=[2],
    state_names={"overload": ["no", "yes"], "heatwave": ["no", "yes"]},
)

# P(powercut | heatwave, overload)
# columns order: (H=no,A=no), (H=no,A=yes), (H=yes,A=no), (H=yes,A=yes)
cpd_powercut = TabularCPD(
    variable="powercut",
    variable_card=2,
    values=[
        [0.95, 0.5, 0.8, 0.3],      # P(powercut=no | ...)
        [0.05, 0.5, 0.2, 0.7],      # P(powercut=yes | ...)
    ],
    evidence=["heatwave", "overload"],
    evidence_card=[2, 2],
    state_names={
        "powercut": ["no", "yes"],
        "heatwave": ["no", "yes"],
        "overload": ["no", "yes"],
    },
)

cpd_cancel = TabularCPD(
    variable="cancel",
    variable_card=2,
    values=[
        [0.9, 0.2],                 # P(cancel=no | powercut=no/yes)
        [0.1, 0.8],                 # P(cancel=yes | powercut=no/yes)
    ],
    evidence=["powercut"],
    evidence_card=[2],
    state_names={"cancel": ["no", "yes"], "powercut": ["no", "yes"]},
)

model.add_cpds(cpd_heatwave, cpd_overload, cpd_powercut, cpd_cancel)

print("Model check:", model.check_model())

Model check: True


### 2. Exact inference with VariableElimination

Compute $P(\text{heatwave} \mid \text{cancel}=\text{yes})$.  
This must match the hand calculation from Exercise 4 (≈ 0.5367).

In [2]:
infer = VariableElimination(model)
result = infer.query(variables=["heatwave"], evidence={"cancel": "yes"})
print(result)
print()
print("P(heatwave = yes | cancel = yes) =", round(result.values[1], 4))

+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(no)  |          0.4633 |
+---------------+-----------------+
| heatwave(yes) |          0.5367 |
+---------------+-----------------+

P(heatwave = yes | cancel = yes) = 0.5367


### 3. Approximate inference – Rejection Sampling (10 000 samples)

Estimate the same probability by rejection sampling.

In [3]:
sampler = BayesianModelSampling(model)
samples = sampler.rejection_sample(
    evidence=[State("cancel", "yes")],
    size=10000,
    show_progress=False,
)

est = (samples["heatwave"] == "yes").mean()
print("Number of accepted samples:", len(samples))
print("Estimated P(heatwave = yes | cancel = yes) =", round(est, 4))

Number of accepted samples: 10000
Estimated P(heatwave = yes | cancel = yes) = 0.5452


### 4. Comparison

The exact probability obtained by variable elimination is approximately **0.5367**.  
The rejection-sampling estimate with 10 000 samples is very close to this value (0.5452 in this run). Any small difference is ordinary Monte-Carlo variation that shrinks as the number of samples increases.